# C7-cnn-transfer — Practice p15 — Solution

The hand trace applies the stride-two stem and one halving at each of `layer2`, `layer3`, and `layer4`; adaptive pooling then removes the remaining 6×6 grid.

In [ ]:
# Cache pin (course convention, plan 009): pretrained weights live in the repo's
# gitignored reference/cache/ -- resolve it from the repo root BEFORE importing torch.
import os, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
             if (p / "pyproject.toml").exists())
os.environ["TORCH_HOME"] = str(_root / "reference" / "cache" / "torch")

import torch
import torch.nn as nn
from torchvision.models import resnet50, ResNet50_Weights

# float32 register (course exception): pretrained resnet50 is a float32 artifact.
# No float64 default here; inputs are cast .to(torch.float32) at the model
# boundary; repeat float32 forwards are bit-identical.
SEED = 20260804

model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
model.eval()
assert next(model.parameters()).dtype == torch.float32

shapes_hand = {
    "conv1": (1, 64, 96, 96),
    "maxpool": (1, 64, 48, 48),
    "layer1": (1, 256, 48, 48),
    "layer2": (1, 512, 24, 24),
    "layer3": (1, 1024, 12, 12),
    "layer4": (1, 2048, 6, 6),
    "avgpool": (1, 2048, 1, 1),
    "fc": (1, 1000),
}


In [ ]:
torch.manual_seed(SEED)
x192 = torch.randn(1, 3, 192, 192).to(torch.float32)
seen = {}
cur = x192
with torch.inference_mode():
    for name, child in model.named_children():
        if name == "fc":
            cur = torch.flatten(cur, 1)
        cur = child(cur)
        if name in ("conv1", "maxpool", "layer1", "layer2", "layer3", "layer4", "avgpool", "fc"):
            seen[name] = tuple(cur.shape)
all_match = seen == shapes_hand


In [ ]:
n_blocks = tuple(len(getattr(model, f"layer{i}")) for i in (1, 2, 3, 4))
n_body_convs = 3 * sum(n_blocks)
depth_50 = n_body_convs + 2


In [ ]:
# Convs: 128*512 + 128*128*9 + 512*128 = 278,528.
hand_convs = 278_528
# BatchNorm: 2*(128 + 128 + 512) = 1,536.
hand_bn = 1_536
hand_total = 280_064


In [ ]:
torch_total = sum(p.numel() for p in model.layer2[3].parameters())
count_gap = abs(hand_total - torch_total)


### Answer check

In [ ]:
assert all_match
assert n_blocks == (3, 4, 6, 3)
assert (n_body_convs, depth_50) == (48, 50)
assert hand_convs + hand_bn == hand_total == 280_064
assert torch_total == 280_064 and count_gap == 0
